<a href="https://colab.research.google.com/github/SuryamaheshPenke/Gen_AI-Notes/blob/main/Class_5_Prompt_Design_Patterns_and_Frameworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Class 5 - Prompt Design Patterns and Frameworks**
### Module 1: Prompt Engineering & LLM Mastery

---

## **Agenda**

- **PART 1 - The 4-Part Prompt:** what the structure is, why it works, what each component contributes
- **PART 2 - Zero-Shot vs Few-Shot:** when examples help, how many to use, how to pick them
- **PART 3 - Persona and Tone Control:** system prompts that define model behaviour
- **PART 4 - Output Format Control:** prompting for JSON, bullet lists, code, and tables
- **PART 5 - Code Generation Prompting:** prompting for code generation and review
- **Code Demos:** 6 runnable demos, one concept each
- **Key Takeaways + Quick Check**

---

## **PART 1 - The 4-Part Prompt**

### Where This Came From

In 2020-2022, most people used LLMs by typing a single sentence and hoping for the best. The model would respond - but often with the wrong format, wrong depth, wrong assumption about who was asking. Responses were inconsistent. The same question asked twice would produce structurally different answers.

Researchers and practitioners noticed that model quality improved dramatically when prompts gave the model more context about what it was supposed to do. By 2023, several prompt frameworks had converged around the same four ingredients: what role the model should play, what task it should do, what background information it needs, and what the output should look like.

This is not a rigid rule. It is a checklist that prevents the most common reasons prompts fail.

---

### Role

**Simple**

Tell the model who it is for this task. A doctor and a teacher both know what "explain this" means, but they explain things differently - different vocabulary, different assumed background, different tone. Giving the model a role anchors its entire response.

**Technical**

The role component conditions the model's output distribution by activating patterns from training data associated with that role. A prompt beginning with "You are a senior software engineer" shifts probability mass toward technical vocabulary, code-oriented responses, and engineering conventions. This is not instruction-following in a rule-based sense - it is distributional steering. The effect is strongest when the role is specific ("senior software engineer specialising in distributed systems") rather than generic ("an expert").

---

### Task

**Simple**

State exactly what you want done. Not "help me with my email" - "rewrite this email to sound more formal and cut it to under 100 words." The model cannot read your mind. Vague tasks produce vague outputs.

**Technical**

The task component specifies the operation the model should perform. It should include: the action verb (summarise, rewrite, classify, extract, generate), the object of the action, and any constraints (length, scope, exclusions). Underspecified tasks cause the model to infer the most probable completion from training data, which may not match what you need. Overspecified tasks can constrain the model into awkward phrasing. The goal is precise, not exhaustive.

---

### Context

**Simple**

Give the model the background it needs but does not have. If you are asking it to respond to a customer complaint, paste the complaint. If you are asking it to write a feature spec, describe the product. Without context, the model invents plausible-sounding details - which is how hallucinations happen.

**Technical**

Context is the grounding component. It provides the model with information that reduces reliance on parametric knowledge (facts baked into weights during training) in favour of in-context information. Without context, the model generates based on generalised priors from training - which is how hallucinations happen. In Module 3 (starting Class 21) we will build full Retrieval-Augmented Generation (RAG) systems that automate this context-injection step; for now, understand that the context field in your prompt is doing the same job manually.

---

### Format

**Simple**

Tell the model what the output should look like. A bullet list? A table? A JSON object? Plain prose? Without this, the model picks whatever format seems most common for the task - and that guess is often wrong for your use case.

**Technical**

The format component specifies the structural and presentational constraints on the output. This controls: output schema (JSON, XML, markdown), rendering target (plain text for APIs, markdown for web), length bounds, section headings, and response persona voice. Format constraints are resolved last in the generation process, which means they can conflict with task constraints (e.g., "be comprehensive" + "respond in 50 words"). When constraints conflict, models typically satisfy the task constraint at the expense of the format constraint - so be explicit about priority.

---

### Before/After: The 4-Part Structure in Practice

**Before (no structure):**
```
Help me with a job description.
```
Output: a generic, overly long job description that may not match your role, seniority level, or company tone.

**After (4-part structure):**
```
Role: You are a technical recruiter at a mid-size SaaS company.
Task: Write a job description for a Senior Backend Engineer role.
Context: The role is on the payments team, requires 5+ years with Python and distributed systems, and reports to the VP of Engineering. We are a Series B company, 200 people, remote-first.
Format: Use these sections: About the Role, What You Will Do (5 bullets), What We Are Looking For (5 bullets), Nice to Have (3 bullets). Keep it under 400 words.
```
Output: precise, scannable, on-brand, exactly the right length.

In [ ]:
# Demo 1 - The 4-Part Prompt: Before vs After
# Install if needed: pip install openai

import openai
import os

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def call_model(prompt, label):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )
    print(f"\n{'='*60}")
    print(f"PROMPT: {label}")
    print(f"{'='*60}")
    print(response.choices[0].message.content)

# BEFORE: vague, unstructured
vague_prompt = "Help me write a job description."

# AFTER: 4-part structure
structured_prompt = """Role: You are a technical recruiter at a mid-size SaaS company.
Task: Write a job description for a Senior Backend Engineer role.
Context: The role is on the payments team, requires 5+ years with Python and distributed systems, reports to VP of Engineering. Series B company, 200 people, remote-first.
Format: Sections: About the Role, What You Will Do (4 bullets), What We Are Looking For (4 bullets). Under 250 words. Plain text, no markdown."""

call_model(vague_prompt, "BEFORE - No Structure")
call_model(structured_prompt, "AFTER - 4-Part Structure")

**What to look at in the output:**

The BEFORE response will likely default to a generic template with placeholder text or make assumptions about the company and role. The AFTER response will follow the exact sections, stay within the word count, and reflect the payments/distributed systems context. The structure did not constrain creativity - it directed it.

---

## **PART 2 - Zero-Shot vs Few-Shot**

### Where This Came From

In the GPT-3 paper (Brown et al., 2020), the authors demonstrated something unexpected: large language models could perform tasks they had never been explicitly trained on, just by seeing a few examples in the prompt. They called this "in-context learning." The model was not being retrained - it was adapting based purely on the pattern shown in the prompt.

This discovery separated two distinct prompting strategies: zero-shot (no examples, just instructions) and few-shot (examples included). Understanding when each strategy works - and why - is one of the most practically useful things in prompt engineering.

---

### Zero-Shot Prompting

**Simple**

You describe what you want, and the model figures it out. No examples needed. This works for tasks the model has seen enough of in training to generalise from: translation, summarisation, basic classification, code explanation.

**Technical**

In zero-shot prompting, the model relies entirely on its parametric knowledge - the patterns compressed into its weights during pretraining and fine-tuning. Zero-shot works well when: (1) the task is well-represented in training data, (2) the task schema is unambiguous from description alone, and (3) the output format is standard. It fails when the task requires a specific schema, domain-specific conventions, or output style that differs from the model's default. Instruction-tuned models (GPT-4, Claude, Gemini) perform significantly better zero-shot than base models because they were explicitly trained to follow instructions without examples.

---

### Few-Shot Prompting

**Simple**

You show the model 2-5 examples of input and the exact output you want. The model learns the pattern from your examples and applies it to the new input. This is useful when the output format is unusual, when you need consistent structure, or when the task has domain-specific rules that are hard to describe in words.

**Technical**

Few-shot prompting uses in-context learning: the model conditions its output distribution on the demonstrated input-output pairs. Unlike fine-tuning, no weight updates occur - the examples exist only in the context window. The mechanism is pattern matching over the attention mechanism: the model attends to the structural relationship between inputs and outputs in your examples and applies the same mapping. Key variables: number of examples (2-5 is the practical sweet spot for most tasks), example quality (incorrect examples actively degrade performance), and example diversity (covering edge cases prevents the model from overfitting to a narrow pattern).

---

### When to Use Each

**Simple**

Use zero-shot when the task is straightforward and you can describe it clearly in words. Use few-shot when: the output has an unusual structure, you need exact consistency across many calls, or you have tried zero-shot and the format keeps coming out wrong.

**Technical**

Decision criteria for zero-shot vs few-shot:
- **Task familiarity:** common NLP tasks (summarise, translate, classify sentiment) - zero-shot. Custom schemas, proprietary formats, domain jargon - few-shot.
- **Output variability:** if zero-shot produces structurally inconsistent outputs across runs, few-shot provides an implicit format lock.
- **Token cost:** each example adds tokens. At scale (millions of calls), fewer examples reduce cost. Start with zero-shot, add examples only when quality metrics justify the cost.
- **Example selection:** examples should be representative, not just easy. Include at least one hard case or edge case the model is likely to encounter. Randomly sampled examples from your target distribution outperform hand-picked "perfect" examples in most benchmarks.

---

### Before/After: Zero-Shot vs Few-Shot for Classification

**Before (zero-shot, inconsistent output):**
```
Classify this support ticket: "I was charged twice for my subscription."
```
Output might be: "billing issue", "Billing", "BILLING_DUPLICATE", "This is a billing problem" - no consistent format.

**After (few-shot, consistent output):**
```
Classify support tickets into exactly one category.

Input: "My app keeps crashing on launch."
Category: BUG

Input: "How do I export my data?"
Category: HOW_TO

Input: "I was charged twice for my subscription."
Category:
```
Output: `BILLING` - and it will be `BILLING` every time.

In [ ]:
# Demo 2 - Zero-Shot vs Few-Shot Classification

import openai
import os

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

tickets = [
    "My app keeps crashing whenever I open settings.",
    "How do I add a second user to my account?",
    "I was charged twice last month.",
    "The export button does nothing when I click it."
]

zero_shot_template = "Classify this support ticket into a category: \"{ticket}\""

few_shot_template = """Classify support tickets. Respond with ONLY the category label.

Input: "My password reset email never arrived."
Category: ACCOUNT_ACCESS

Input: "How do I connect my calendar?"
Category: HOW_TO

Input: "The dashboard takes 30 seconds to load."
Category: PERFORMANCE

Input: "{ticket}"
Category:"""

print("ZERO-SHOT vs FEW-SHOT - Support Ticket Classification")
print("="*60)

for ticket in tickets:
    zs_resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": zero_shot_template.format(ticket=ticket)}],
        max_tokens=20
    )
    fs_resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": few_shot_template.format(ticket=ticket)}],
        max_tokens=10
    )
    print(f"\nTicket: {ticket}")
    print(f"  Zero-shot: {zs_resp.choices[0].message.content.strip()}")
    print(f"  Few-shot:  {fs_resp.choices[0].message.content.strip()}")

**What to look at in the output:**

Zero-shot responses will vary in format - some may be full sentences, some capitalized, some with explanations. Few-shot responses will be terse, consistent category labels every time. Run this multiple times and watch the zero-shot output format drift while few-shot stays stable.

---

## **PART 3 - Persona and Tone Control**

### Where This Came From

When OpenAI released the Chat Completions API in 2022, they introduced a three-role message structure: `system`, `user`, and `assistant`. The system role was new - it was a privileged channel that sits above the conversation and defines how the model should behave throughout the entire interaction.

Before system prompts existed, developers had to put persona instructions at the start of every user message. This was brittle: the model could drift from the persona over long conversations, and users could easily override it. System prompts solved this by making persona instructions persistent and harder to override from the user turn.

System prompts are now the standard mechanism for building consistent AI products - from customer service bots that always sound on-brand to coding assistants that never break character.

---

### Persona

**Simple**

A persona tells the model who it is, not just what to do. A customer service bot for a bank should sound different from a customer service bot for a gaming company. The persona sets the vocabulary, the warmth level, the formality, and the assumed relationship with the user.

**Technical**

A persona in a system prompt conditions the model's output distribution across the entire conversation, not just the next turn. It defines: (1) **identity** - the model's name, role, and relationship to the user; (2) **knowledge scope** - what the model knows and does not know ("you only have information about our product catalog"); (3) **behavioural constraints** - what the model will and will not do ("never discuss competitor pricing"); (4) **conversational defaults** - how it responds to ambiguity, off-topic questions, or adversarial inputs. A well-written persona reduces variance across thousands of user interactions and is the primary tool for building predictable AI products.

---

### Tone Control

**Simple**

Tone is the emotional register of the response. Formal vs casual. Encouraging vs neutral. Concise vs thorough. You control tone explicitly by describing it ("be concise and direct") or by example (show what a response in that tone looks like).

**Technical**

Tone is a composite of several output properties: lexical formality (word choice and register), sentence complexity (average clause depth), hedging frequency (use of "may", "might", "could be"), and affective markers (expressions of encouragement, empathy, or urgency). Models respond to explicit tone descriptors, but are more reliably constrained by specific behavioural rules: "use sentences under 20 words" is more reliable than "be concise". Tone drift occurs in long conversations when user messages pull the model toward the user's own register - system prompt reinforcement (repeating key tone constraints in the system prompt) reduces this.

---

### Before/After: Persona and Tone

**Before (no persona):**
```
User: My order hasn't arrived.
Model: I'm sorry to hear that. Your order may be delayed due to various factors including shipping carrier issues, weather, or high demand periods.
```
Generic, corporate, no action.

**After (with persona and tone):**
```
System: You are Maya, a support specialist at Bloom Florists. You are warm, direct, and solution-focused. When a customer reports a missing order, immediately offer to track it and provide a resolution timeline. Never say "may be delayed" without giving a specific next step.
User: My order hasn't arrived.
Model: Hi! I'm really sorry about that - let me look into it right now. Can you share your order number? I'll pull up the tracking and give you a clear update within 2 minutes.
```
Specific name, immediate action, concrete next step.

In [ ]:
# Demo 3 - Persona and Tone via System Prompt

import openai
import os

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

user_message = "My order hasn't arrived and it was supposed to be here yesterday."

personas = [
    {
        "label": "No Persona (default)",
        "system": None
    },
    {
        "label": "Formal Corporate Support",
        "system": "You are a customer support representative for a logistics company. Respond formally. Acknowledge the issue, explain possible causes, and offer to escalate. Keep responses under 80 words."
    },
    {
        "label": "Friendly Direct Support (Maya)",
        "system": "You are Maya, a support specialist at Bloom Florists. You are warm, direct, and solution-focused. Always ask for the order number immediately. Never say 'may be delayed' without giving a specific next step. Keep responses under 60 words."
    }
]

for persona in personas:
    messages = []
    if persona["system"]:
        messages.append({"role": "system", "content": persona["system"]})
    messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        max_tokens=120
    )
    print(f"\n--- {persona['label']} ---")
    print(response.choices[0].message.content)

**What to look at in the output:**

Compare the first response (no system prompt) against the third (Maya persona). The words used, the action offered, and the tone are substantially different - not because the task changed, but because the persona conditioned the entire response. Notice how the formal persona tends toward passive constructions while the friendly persona leads with action verbs.

---

## **PART 4 - Output Format Control**

### Where This Came From

The most common failure mode in production LLM applications is not wrong content - it is wrong format. An application expects a JSON object, the model returns a JSON object wrapped in a markdown code block. A pipeline expects a list, the model returns a numbered paragraph. Format failures break downstream parsing and require either expensive retry loops or fragile string cleaning code.

Format control is the discipline of making the model's output structurally reliable. This matters more as you move from one-off prompts to automated pipelines where 1 in 100 malformed responses breaks a workflow that processes 10,000 records per day.

Some of this can be solved at the infrastructure level using native JSON mode - we will cover that in Class 10. For now, all format control happens at the prompt level.

---

### Prompting for JSON

**Simple**

Ask for JSON explicitly, and show the exact schema you need. Do not assume the model knows what fields you want. Tell it every key, what type each value should be, and what to put there when data is missing.

**Technical**

When prompting for JSON output: (1) specify the schema inline using a JSON example with commented field descriptions, or as explicit field definitions; (2) specify what to do on missing data (null, empty string, or omit the key entirely); (3) instruct the model to return raw JSON without markdown fencing (unless your parser strips it). Models trained with RLHF tend to add markdown formatting by default - the instruction "respond with raw JSON only, no code block" is necessary to prevent this. For guaranteed JSON validity in production, combine prompt-level instructions with response_format: {type: "json_object"} (OpenAI) or Pydantic validation as a post-processing step.

---

### Prompting for Other Formats

**Simple**

Bullet lists: say how many bullets, max length per bullet, and whether sub-bullets are allowed. Tables: name every column and say what each column contains. Code: specify language, whether to include docstrings, and whether to include a usage example. The more specific your format instruction, the more consistent the output.

**Technical**

Format precision follows a quality-specificity tradeoff: vague format instructions ("give me a table") produce outputs that look like tables but vary in column count, ordering, and null handling across runs. Precise instructions ("generate a markdown table with columns: Field, Type, Description, Required - one row per JSON key") produce consistent, machine-parseable outputs. For structured outputs, always specify: (1) the exact structure type, (2) naming conventions for fields/columns/headers, (3) how to handle absent or null values, (4) whether to include preamble text or respond with the structure only.

In [ ]:
# Demo 4 - Output Format Control: JSON, Table, Bullet List

import openai
import json
import os

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

raw_info = """Priya Sharma, Senior Engineer, 6 years experience, Python and Go,
previously at Flipkart and Razorpay, looking for staff-level roles, open to remote."""

# Format 1: JSON
json_prompt = f"""Extract candidate information from the text below and return ONLY a raw JSON object.
No markdown, no code blocks, just the JSON.

Required schema:
{{
  "name": "full name as string",
  "title": "current title as string",
  "years_experience": "number",
  "skills": ["list of strings"],
  "previous_companies": ["list of strings"],
  "target_level": "string",
  "remote_preference": "string"
}}

Text: {raw_info}"""

# Format 2: Markdown Table
table_prompt = f"""Extract candidate information from the text below.
Respond with ONLY a markdown table. Columns: Field | Value.
No preamble, no explanation.

Text: {raw_info}"""

# Format 3: Bullet Summary
bullet_prompt = f"""Summarise this candidate in exactly 4 bullet points.
Each bullet: max 12 words. Start each bullet with a category label in bold (e.g. **Role:**).
No preamble.

Text: {raw_info}"""

for label, prompt in [("JSON", json_prompt), ("Table", table_prompt), ("Bullets", bullet_prompt)]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200
    )
    output = resp.choices[0].message.content
    print(f"\n{'='*50}")
    print(f"FORMAT: {label}")
    print(f"{'='*50}")
    print(output)
    if label == "JSON":
        try:
            parsed = json.loads(output)
            print(f"\n[JSON valid - {len(parsed)} keys parsed]")
        except json.JSONDecodeError as e:
            print(f"\n[JSON parse error: {e}]")

**What to look at in the output:**

The same source text produces three structurally different outputs. Check the JSON section - if the parse succeeds, the model followed the schema instruction. If it fails, the model likely wrapped the JSON in a code block; add "no markdown fencing" to the instruction to fix it. Notice how the bullet output stays within the label format when constraints are specific.

In [ ]:
# Demo 5 - Full 4-Part Prompt: All Components Together
# Task: generate a structured interview brief from a job description

import openai
import os

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

job_description = """
We are hiring a Senior Data Engineer for our ML Platform team.
The role involves building and maintaining data pipelines,
working with Apache Spark and dbt, and collaborating with
ML engineers on feature engineering infrastructure.
Required: 5+ years experience, strong Python, SQL, and experience
with cloud data platforms (AWS or GCP).
"""

# 4-part prompt assembled
role = "You are a technical recruiter at a Series B startup preparing an interview brief for a hiring manager."

task = "Generate a structured interview brief for the following job description."

context = f"The hiring manager is technical (former engineer). They need to run a 45-minute first-round interview. The company has used Databricks and is migrating to dbt. Compensation band is $150k-$180k.\n\nJob description:\n{job_description}"

format_instruction = """Format the brief as follows (use these exact headings):
**Role Summary** - 2 sentences max
**Must-Have Skills** - 4 bullet points, be specific
**Red Flags to Watch For** - 3 bullet points
**Suggested Interview Questions** - 5 questions, each under 20 words
**Hiring Signal** - one sentence: what a strong yes looks like

Respond with markdown only. No preamble."""

# Assemble full prompt
full_prompt = f"""{role}

{task}

Context: {context}

Output format: {format_instruction}"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": full_prompt}],
    max_tokens=400
)

print("FULL 4-PART PROMPT OUTPUT")
print("="*60)
print(response.choices[0].message.content)

**What to look at in the output:**

This response demonstrates all four components working together. The role (recruiter) shapes the vocabulary. The task (interview brief) determines the content type. The context (Databricks to dbt migration, compensation band) surfaces in the questions and red flags. The format instruction produces the exact headings in the exact order. Remove any one component and observe what degrades.

---

## **Code Generation Prompting**

Code generation is one of the highest-value uses of LLMs in practice, but it requires a different prompting approach than prose tasks. Code either runs or it does not - approximate output is often worse than no output. This section covers how to prompt for code generation and code review. Class 11 extends these patterns to multimodal contexts (prompting with code screenshots and diagrams).

---

### Prompting for Code Generation

**Simple**

The best code generation prompts read like a precise engineering spec. State the language and version, the function signature with type annotations, the exact input/output contract with examples, what edge cases to handle, and what NOT to use. Vague prompts like "write a function to parse emails" produce code that works for the obvious case and breaks on everything else.

**Technical**

High-yield code generation prompt structure: (1) Language and version - "Python 3.11"; (2) Function signature with type annotations; (3) Exact input/output contract with at least one example; (4) Error handling - what to raise vs. what to return None for; (5) Performance constraints if relevant; (6) Excluded approaches - "do not use regex", "do not make network calls"; (7) Available dependencies. Including a minimal failing test case in the prompt significantly improves output correctness because it forces the model to reason about the boundary condition before writing the implementation.

---

### Prompting for Code Review

**Simple**

"Review this code" is too broad and produces generic output. Specify the review axis: security, correctness, performance, readability, or maintainability. "Review this code for SQL injection vulnerabilities only" gives you a focused, usable review. You can also frame the perspective: "review this as the engineer who will maintain it in six months."

**Technical**

Code review prompts that produce actionable output: (1) Specify the review axis explicitly; (2) Provide surrounding context - what this integrates with; (3) Ask for severity ratings (critical / warning / suggestion); (4) Request inline references by line number; (5) Ask for a fix for each finding, not just identification. For large files, send function-by-function rather than dumping everything into one prompt.

In [ ]:
# Demo 6 - Code Generation Prompt
# A well-structured prompt that produces a usable, tested function

import openai
import os

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

code_gen_prompt = """Language: Python 3.11
Task: Write a function called `parse_email_parts` that extracts the username and domain from an email address.

Function signature:
    def parse_email_parts(email: str) -> dict:

Input/output contract:
    Input: a string that may or may not be a valid email
    Output: {"username": str, "domain": str} if valid, raise ValueError if not

Edge cases to handle:
    - Missing @ symbol -> ValueError
    - Multiple @ symbols -> ValueError
    - Empty string -> ValueError
    - Valid: "user@example.com" -> {"username": "user", "domain": "example.com"}

Constraints:
    - Do not use regex
    - No third-party libraries
    - Include a docstring
    - Include 3 example calls in a comment at the bottom"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": code_gen_prompt}],
    max_tokens=400
)

print(response.choices[0].message.content)

---

# **Key Takeaways**

1. The 4-part prompt (Role + Task + Context + Format) is not a rigid template - it is a checklist of what information the model needs to produce a useful, consistent output. Each part solves a specific failure mode: role anchors vocabulary and depth, task specifies the operation, context prevents hallucination, format makes output parseable.

2. Zero-shot works well for common tasks the model has seen in training. Few-shot is the fix when you need a specific output structure the model cannot infer from a description alone - 2-5 high-quality examples consistently outperform longer zero-shot instructions for format-sensitive tasks.

3. System prompts are the mechanism for persistent persona - they condition every response in the conversation, not just the next turn. They define identity, knowledge scope, behavioural constraints, and tone defaults all at once.

4. Tone control is more reliable with specific behavioural rules ("use sentences under 20 words") than with abstract descriptors ("be concise"). Abstract descriptors give the model too much interpretive latitude.

5. Format instructions must specify structure type, field/column names, null handling, and whether to include preamble text. Vague format instructions ("give me a table") produce inconsistent structures that break parsing pipelines.

6. Few-shot example quality matters more than quantity. One wrong example actively degrades output. Cover edge cases in your example set - do not pick only easy or "perfect" examples.

---

# **Quick Check**

1. You prompt a model with: "You are a financial analyst. Summarise this earnings report in 3 bullets for a non-technical board audience. [report text]." Which of the 4 parts is missing, and how would that affect the output?

2. A classmate says: "I always use few-shot because it gets better results." What is the cost tradeoff they are ignoring, and when does zero-shot match or beat few-shot performance?

3. You are building a support chatbot for a children's learning app. Write the core persona instruction (2-3 sentences) that would define tone, vocabulary level, and one behavioural constraint.

4. Your pipeline calls the model 50,000 times a day and expects a JSON object with 5 fields. On about 300 calls per day, the model wraps the JSON in a markdown code block and the parser fails. What is the prompt-level fix, and what infrastructure-level fix would you add in production?

5. You try zero-shot to classify customer feedback into 6 custom categories (FEATURE_REQUEST, BUG, PRAISE, BILLING, CANCELLATION, OTHER) and the model keeps returning "positive" or "negative" instead. Why is this happening, and what is the minimum change to fix it?

6. You have 20 few-shot examples for a task. You test 3 examples vs 10 examples vs 20 examples and the quality plateaus after 4. What does this tell you about the practical optimal for this task, and why does adding more examples not help?

7. A developer on your team says the format instruction "be thorough but concise" is causing unpredictable output lengths. Rewrite this as two specific constraints that would produce consistent output length.

## **Quick Check - Answers**

---

*Attempt the questions above before reading these answers.*

---

**1.** The format part is missing. The prompt has persona (financial analyst), task (summarise in 3 bullets), and context ([report text]), but specifies no output format beyond "3 bullets." Without explicit format constraints, the model may use different bullet styles, add headers, or vary length unpredictably across calls.

**2.** Few-shot examples consume tokens on every call - at 50K calls per day, even 3 examples add significant cost and latency at scale. Zero-shot matches or beats few-shot when the task is well-represented in training data (sentiment, translation, JSON formatting) and when the instruction alone is unambiguous.

**3.** Example: "You are a friendly assistant for a children's learning app designed for ages 6-10. Use simple vocabulary - avoid any word a 4th grader would not know. Never discuss violence, adult topics, or anything unrelated to learning and encouragement."

**4.** Prompt-level fix: add "Return only the JSON object. No markdown, no code fences, no explanation." Infrastructure fix: add a parser wrapper that strips markdown code blocks (` ```json...``` `) before JSON parsing, so formatting variation does not cause failures in production.

**5.** The model defaults to sentiment classification because that is the most common text classification task in its training data. The minimum fix: list the exact category names and instruct it to return only one of them - "Classify into exactly one of: FEATURE_REQUEST, BUG, PRAISE, BILLING, CANCELLATION, OTHER. Return only the category name."

**6.** The practical optimal is 4-5 examples. Adding more does not help because the model has already learned the pattern - additional examples become redundant given what it already learned from pre-training. More examples also increase prompt length and cost without improving quality once the pattern is established.

**7.** Instead of "be thorough but concise," use: "Write exactly 3 sentences per answer" and "Do not include background context or definitions unless the question asks for them." Specific numerical and structural constraints produce consistent output length; vague qualitative instructions do not.